# 0. Import

In [1]:
!pip install wandb -q

In [2]:
import json
import wandb
import numpy as np
import torch

from torch.utils.data import Dataset, DataLoader

In [3]:
import os
import wandb
from kaggle_secrets import UserSecretsClient

# Fetch the secret token safely
user_secrets = UserSecretsClient()
os.environ["WANDB_API_KEY"] = user_secrets.get_secret("WANDB_API_KEY")

# Log into WandB
wandb.login()

wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from WANDB_API_KEY.
wandb: Currently logged in as: hariswarsamasi (hariswarsamasi-indian-institute-of-technology-madras) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


True

# 1. Making the Modality

In [4]:
def load_skeleton(trial_path):

    prediction_dir = Path(trial_path) / "predictions"

    json_files = sorted(prediction_dir.glob("*.json"))

    frames = []

    for json_file in json_files:

        with open(json_file, "r") as f:
            data = json.load(f)

        # One person per frame based on our inspection
        person = data[0]

        keypoints = np.asarray(
            person["keypoints"],
            dtype=np.float32
        )

        frames.append(keypoints)

    if len(frames) == 0:
        return None

    return np.stack(frames)

## 1. Building the training index

In [5]:
def temporal_resample(x, target_frames=64):

    T = x.shape[0]

    if T == target_frames:
        return x.astype(np.float32)

    old_indices = np.linspace(
        0,
        T - 1,
        T
    )

    new_indices = np.linspace(
        0,
        T - 1,
        target_frames
    )

    output = np.empty(
        (target_frames, x.shape[1], x.shape[2]),
        dtype=np.float32
    )

    for joint in range(x.shape[1]):
        for coord in range(x.shape[2]):

            output[:, joint, coord] = np.interp(
                new_indices,
                old_indices,
                x[:, joint, coord]
            )

    return output

In [6]:
def normalize_skeleton(x):
    """
    x: (T, 17, 3)

    Makes skeleton coordinates root-relative.
    Joint 0 is treated as the root.
    """

    x = x.copy()

    # Root-relative coordinates
    root = x[:, 0:1, :]
    x = x - root

    return x.astype(np.float32)

In [7]:
import json
import numpy as np
import torch
from pathlib import Path
from torch.utils.data import Dataset, DataLoader

def temporal_resample(x, target_frames=64):
    """
    Resample the complete sequence to exactly target_frames.
    """

    T = x.shape[0]

    if T == target_frames:
        return x.astype(np.float32)

    if T == 1:
        return np.repeat(
            x,
            target_frames,
            axis=0
        ).astype(np.float32)

    old_indices = np.linspace(
        0,
        T - 1,
        T
    )

    new_indices = np.linspace(
        0,
        T - 1,
        target_frames
    )

    output = np.empty(
        (target_frames, x.shape[1], x.shape[2]),
        dtype=np.float32
    )

    for joint in range(x.shape[1]):

        for coord in range(x.shape[2]):

            output[:, joint, coord] = np.interp(
                new_indices,
                old_indices,
                x[:, joint, coord]
            )

    return output.astype(np.float32)

import json
import numpy as np
import torch
from pathlib import Path
from torch.utils.data import Dataset, DataLoader

import json
import numpy as np
import torch
from pathlib import Path
from torch.utils.data import Dataset


class SkeletonDataset(Dataset):

    def __init__(self, df, sequence_length=64):
        self.df = df.reset_index(drop=True)
        self.sequence_length = sequence_length

    def load_skeleton(self, path):

        prediction_dir = Path(path) / "predictions"

        json_files = sorted(prediction_dir.glob("*.json"))

        frames = []

        for json_file in json_files:

            with open(json_file, "r") as f:
                data = json.load(f)

            if len(data) == 0:
                continue

            # One person per frame
            person = data[0]

            keypoints = np.asarray(
                person["keypoints"],
                dtype=np.float32
            )

            frames.append(keypoints)

        if len(frames) == 0:
            return None

        return np.stack(frames)


    def normalize_skeleton(self, x):

        # x: (T, 17, 3)

        x = x.copy()

        # Root-relative coordinates
        root = x[:, 0:1, :]

        x = x - root

        return x


    def temporal_resample(self, x):

        T = x.shape[0]

        if T == self.sequence_length:
            return x.astype(np.float32)

        # Very short sequence
        if T == 1:
            return np.repeat(
                x,
                self.sequence_length,
                axis=0
            ).astype(np.float32)

        old_indices = np.linspace(
            0,
            T - 1,
            T
        )

        new_indices = np.linspace(
            0,
            T - 1,
            self.sequence_length
        )

        output = np.empty(
            (
                self.sequence_length,
                x.shape[1],
                x.shape[2]
            ),
            dtype=np.float32
        )

        for joint in range(x.shape[1]):

            for coord in range(x.shape[2]):

                output[:, joint, coord] = np.interp(
                    new_indices,
                    old_indices,
                    x[:, joint, coord]
                )

        return output


    def __len__(self):
        return len(self.df)


    def __getitem__(self, idx):

        row = self.df.iloc[idx]

        skeleton = self.load_skeleton(
            row["skeleton"]
        )

        if skeleton is None:

            skeleton = np.zeros(
                (
                    self.sequence_length,
                    17,
                    3
                ),
                dtype=np.float32
            )

        # --------------------------------
        # 1. Root-relative coordinates
        # --------------------------------

        skeleton = self.normalize_skeleton(
            skeleton
        )

        # --------------------------------
        # 2. Temporal resampling
        # --------------------------------

        skeleton = self.temporal_resample(
            skeleton
        )

        X = torch.tensor(
            skeleton,
            dtype=torch.float32
        )

        y = torch.tensor(
            row["label"],
            dtype=torch.long
        )

        return X, y

In [8]:
from pathlib import Path
import pandas as pd

TRAIN_DATA = Path(
    "/kaggle/input/datasets/samasiayushman/"
    "small-model-track/Training/Training/data"
)

MODALITIES = [
    "Skeleton",
    "Depth_Color",
    "IR",
    "Thermal",
    "IMU",
    "Radar",
]

# ---------------------------------------------------------
# Build trial index from Skeleton
# ---------------------------------------------------------

records = []

skeleton_root = TRAIN_DATA / "Skeleton"

for action_dir in sorted(skeleton_root.iterdir()):

    if not action_dir.is_dir():
        continue

    action_name = action_dir.name
    label = int(action_name.split("_")[0])

    for user_dir in sorted(action_dir.iterdir()):

        if not user_dir.is_dir():
            continue

        user = user_dir.name

        for trial_dir in sorted(user_dir.iterdir()):

            if not trial_dir.is_dir():
                continue

            records.append({
                "action": action_name,
                "label": label,
                "user": user,
                "trial": trial_dir.name,
                "path": str(trial_dir),
            })


train_df = pd.DataFrame(records)

print("Training trials:", len(train_df))
print("Classes:", train_df["label"].nunique())
print("Users:", train_df["user"].nunique())

train_df.head()

Training trials: 2931
Classes: 40
Users: 18


,action,label,user,trial,path
0,0_Wash_face,0,user16,1-1-1,/kaggle/input/datasets/samasiayushman/small-mo...
1,0_Wash_face,0,user16,1-1-2,/kaggle/input/datasets/samasiayushman/small-mo...
2,0_Wash_face,0,user16,1-1-3,/kaggle/input/datasets/samasiayushman/small-mo...
3,0_Wash_face,0,user18,7-1-1,/kaggle/input/datasets/samasiayushman/small-mo...
4,0_Wash_face,0,user18,7-1-2,/kaggle/input/datasets/samasiayushman/small-mo...


In [9]:
from sklearn.model_selection import GroupShuffleSplit

splitter = GroupShuffleSplit(
    n_splits=1,
    test_size=0.2,
    random_state=42
)

train_idx, val_idx = next(
    splitter.split(
        train_df,
        train_df["label"],
        groups=train_df["user"]
    )
)

train_split = train_df.iloc[train_idx].reset_index(drop=True)
val_split = train_df.iloc[val_idx].reset_index(drop=True)

print("Train:", len(train_split))
print("Validation:", len(val_split))

print("Train users:")
print(sorted(train_split["user"].unique()))

print("\nValidation users:")
print(sorted(val_split["user"].unique()))

Train: 2238
Validation: 693
Train users:
['user17', 'user18', 'user19', 'user20', 'user21', 'user23', 'user24', 'user3', 'user4', 'user5', 'user6', 'user7', 'user8', 'user9']

Validation users:
['user1', 'user16', 'user2', 'user22']


In [10]:
train_dataset = SkeletonDataset(
    train_split.assign(skeleton=train_split["path"]),
    sequence_length=64
)

val_dataset = SkeletonDataset(
    val_split.assign(skeleton=val_split["path"]),
    sequence_length=64
)

print("Train dataset:", len(train_dataset))
print("Val dataset:", len(val_dataset))

Train dataset: 2238
Val dataset: 693


In [11]:
train_loader = DataLoader(
    train_dataset,
    batch_size=32,
    shuffle=True,
    num_workers=0,
    pin_memory=False
)

val_loader = DataLoader(
    val_dataset,
    batch_size=32,
    shuffle=False,
    num_workers=0,
    pin_memory=False
)

In [12]:
X, y = next(iter(train_loader))

print("Train batch X:", X.shape)
print("Train batch y:", y.shape)

X, y = next(iter(val_loader))

print("Val batch X:", X.shape)
print("Val batch y:", y.shape)

Train batch X: torch.Size([32, 64, 17, 3])
Train batch y: torch.Size([32])
Val batch X: torch.Size([32, 64, 17, 3])
Val batch y: torch.Size([32])


In [13]:
dataset =  SkeletonDataset(
    train_split.assign(skeleton=train_split["path"]),
    sequence_length=64
)


X, y = dataset[0]

print("X shape:", X.shape)
print("y:", y)
print("dtype:", X.dtype)

print("Min:", X.min().item())
print("Max:", X.max().item())
print("Mean:", X.mean().item())
print("Std:", X.std().item())

X shape: torch.Size([64, 17, 3])
y: tensor(0)
dtype: torch.float32
Min: -0.568962812423706
Max: 0.809485673904419
Mean: 0.13565883040428162
Std: 0.2951955497264862


---

In [14]:
X, y = train_dataset[0]

print("X shape:", X.shape)
print("y:", y)
print("dtype:", X.dtype)

print("Min:", X.min().item())
print("Max:", X.max().item())
print("Mean:", X.mean().item())
print("Std:", X.std().item())

X shape: torch.Size([64, 17, 3])
y: tensor(0)
dtype: torch.float32
Min: -0.568962812423706
Max: 0.809485673904419
Mean: 0.13565883040428162
Std: 0.2951955497264862


In [15]:
print("Root joint mean:", X[:, 0, :].abs().mean().item())

print( "Root joint max:", X[:, 0, :].abs().max().item())

Root joint mean: 0.0
Root joint max: 0.0


# A. Baseline Model

In [16]:
import torch
import torch.nn as nn


class BiLSTM(nn.Module):

    def __init__(
        self,
        input_size=51,
        hidden_size=128,
        num_layers=2,
        num_classes=40,
        dropout=0.3
    ):

        super().__init__()

        self.lstm = nn.LSTM(
            input_size=input_size,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout if num_layers > 1 else 0,
            bidirectional=True
        )

        # BiLSTM produces 2 * hidden_size
        self.classifier = nn.Sequential(
            nn.Dropout(dropout),
            nn.Linear(hidden_size * 2, num_classes)
        )

    def forward(self, x):

        # x:
        # (batch, 64, 17, 3)

        batch_size = x.size(0)

        # Flatten joints + coordinates
        x = x.reshape(
            batch_size,
            x.size(1),
            -1
        )
        # (B, 64, 51)
        output, (hidden, cell) = self.lstm(x)
        
        # output:
        # (B, 64, 256)
        # Last temporal representation
        x = output[:, -1, :]

        # (B, 256)
        logits = self.classifier(x)

        return logits

In [17]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = BiLSTM(
    input_size=17 * 3,
    hidden_size=128,
    num_layers=2,
    num_classes=40,
    dropout=0.3
).to(device)

print(model)
print("Device:", device)

BiLSTM(
  (lstm): LSTM(51, 128, num_layers=2, batch_first=True, dropout=0.3, bidirectional=True)
  (classifier): Sequential(
    (0): Dropout(p=0.3, inplace=False)
    (1): Linear(in_features=256, out_features=40, bias=True)
  )
)
Device: cpu


In [18]:
X, y = next(iter(train_loader))

X = X.to(device)

with torch.no_grad():
    output = model(X)

print("Input :", X.shape)
print("Output:", output.shape)
X, y = next(iter(train_loader))

print("Train batch X:", X.shape)
print("Train batch y:", y.shape)

X, y = next(iter(val_loader))

print("Val batch X:", X.shape)
print("Val batch y:", y.shape)

Input : torch.Size([32, 64, 17, 3])
Output: torch.Size([32, 40])
Train batch X: torch.Size([32, 64, 17, 3])
Train batch y: torch.Size([32])
Val batch X: torch.Size([32, 64, 17, 3])
Val batch y: torch.Size([32])


In [19]:
criterion = nn.CrossEntropyLoss()

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=1e-3
)

In [20]:
wandb.init(
    project="CIUX",
    name="BiLstm",
    config={
        "model": "BiLSTM",
        "sequence_length": 64,
        "num_keypoints": 17,
        "coordinates": 3,
        "hidden_size": 128,
        "num_layers": 2,
        "dropout": 0.3,
        "batch_size": 32,
        "learning_rate": 1e-3,
        "epochs": 30,
        "optimizer": "Adam",
    }
)

wandb: Tracking run with wandb version 0.26.1
wandb: Run data is saved locally in /kaggle/working/wandb/run-20260901_022526-f2a6cmbz
wandb: Run `wandb offline` to turn off syncing.
wandb: Syncing run BiLstm
wandb: ⭐️ View project at https://wandb.ai/hariswarsamasi-indian-institute-of-technology-madras/CIUX
wandb: 🚀 View run at https://wandb.ai/hariswarsamasi-indian-institute-of-technology-madras/CIUX/runs/f2a6cmbz


In [21]:
epochs = 30

for epoch in range(epochs):

    # ==========================================================
    # TRAIN
    # ==========================================================

    model.train()

    train_loss = 0.0
    train_correct = 0
    train_total = 0

    for X, y in train_loader:

        X = X.to(device)
        y = y.to(device)

        optimizer.zero_grad()

        output = model(X)

        loss = criterion(output, y)

        loss.backward()

        optimizer.step()

        train_loss += loss.item() * X.size(0)

        predictions = output.argmax(dim=1)

        train_correct += (predictions == y).sum().item()
        train_total += y.size(0)

    train_loss /= train_total
    train_accuracy = train_correct / train_total


    # ==========================================================
    # VALIDATION
    # ==========================================================

    model.eval()

    val_loss = 0.0
    val_correct = 0
    val_total = 0

    with torch.no_grad():

        for X, y in val_loader:

            X = X.to(device)
            y = y.to(device)

            output = model(X)

            loss = criterion(output, y)

            val_loss += loss.item() * X.size(0)

            predictions = output.argmax(dim=1)

            val_correct += (predictions == y).sum().item()
            val_total += y.size(0)

    val_loss /= val_total
    val_accuracy = val_correct / val_total


    # ==========================================================
    # PRINT
    # ==========================================================

    print(
        f"Epoch {epoch+1:02d}/{epochs} | "
        f"Train Loss: {train_loss:.4f} | "
        f"Train Acc: {train_accuracy:.4f} | "
        f"Val Loss: {val_loss:.4f} | "
        f"Val Acc: {val_accuracy:.4f}"
    )


    # ==========================================================
    # W&B
    # ==========================================================

    wandb.log({
        "epoch": epoch + 1,

        "train/loss": train_loss,
        "train/accuracy": train_accuracy,

        "val/loss": val_loss,
        "val/accuracy": val_accuracy,

        "learning_rate": optimizer.param_groups[0]["lr"]
    })

Epoch 01/30 | Train Loss: 3.4625 | Train Acc: 0.1180 | Val Loss: 3.1917 | Val Acc: 0.1847
Epoch 02/30 | Train Loss: 3.0941 | Train Acc: 0.1716 | Val Loss: 2.9052 | Val Acc: 0.1847
Epoch 03/30 | Train Loss: 2.8248 | Train Acc: 0.2136 | Val Loss: 2.8476 | Val Acc: 0.1962
Epoch 04/30 | Train Loss: 2.6470 | Train Acc: 0.2475 | Val Loss: 2.5901 | Val Acc: 0.2785
Epoch 05/30 | Train Loss: 2.5428 | Train Acc: 0.2752 | Val Loss: 2.5881 | Val Acc: 0.2554
Epoch 06/30 | Train Loss: 2.4467 | Train Acc: 0.2900 | Val Loss: 2.7058 | Val Acc: 0.2684
Epoch 07/30 | Train Loss: 2.3469 | Train Acc: 0.3105 | Val Loss: 2.5030 | Val Acc: 0.3102
Epoch 08/30 | Train Loss: 2.2374 | Train Acc: 0.3253 | Val Loss: 2.4285 | Val Acc: 0.2915
Epoch 09/30 | Train Loss: 2.2624 | Train Acc: 0.3239 | Val Loss: 2.7159 | Val Acc: 0.2973
Epoch 10/30 | Train Loss: 2.2605 | Train Acc: 0.3105 | Val Loss: 2.4061 | Val Acc: 0.3001
Epoch 11/30 | Train Loss: 2.1935 | Train Acc: 0.3391 | Val Loss: 2.3859 | Val Acc: 0.3420
Epoch 12/3

In [22]:
wandb.finish()

wandb: updating run metadata
wandb: uploading output.log; uploading wandb-summary.json; uploading config.yaml
wandb: uploading output.log; uploading wandb-summary.json
wandb: 
wandb: Run history:
wandb:          epoch ▁▁▁▂▂▂▂▃▃▃▃▄▄▄▄▅▅▅▅▆▆▆▆▇▇▇▇███
wandb:  learning_rate ▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
wandb: train/accuracy ▁▂▃▃▄▄▄▅▅▄▅▅▅▅▆▆▆▆▆▇▆▇▆▇▇▇████
wandb:     train/loss █▇▆▅▅▄▄▄▄▄▄▃▃▃▃▂▂▂▂▂▂▂▃▂▂▁▁▁▁▁
wandb:   val/accuracy ▁▁▁▄▃▄▅▄▅▅▆▄▅▅▆▆▆▅▆▇▇▅▆▇▇▇██▇▇
wandb:       val/loss █▆▅▄▄▄▃▂▄▂▂▃▃▂▂▂▂▃▂▂▂▃▂▁▁▂▁▁▂▂
wandb: 
wandb: Run summary:
wandb:          epoch 30
wandb:  learning_rate 0.001
wandb: train/accuracy 0.48481
wandb:     train/loss 1.55637
wandb:   val/accuracy 0.35642
wandb:       val/loss 2.3398
wandb: 
wandb: 🚀 View run BiLstm at: https://wandb.ai/hariswarsamasi-indian-institute-of-technology-madras/CIUX/runs/f2a6cmbz
wandb: ⭐️ View project at: https://wandb.ai/hariswarsamasi-indian-institute-of-technology-madras/CIUX
wandb: Synced 5 W&B file(s), 0 media file(s), 0 artifact f

---

In [23]:
class SkeletonTestDataset(SkeletonDataset):

    def __getitem__(self, idx):

        row = self.df.iloc[idx]

        skeleton = self.load_skeleton(
            row["skeleton"]
        )

        # No skeleton
        if skeleton is None:
            skeleton = np.zeros(
                (
                    self.sequence_length,
                    17,
                    3
                ),
                dtype=np.float32
            )

        # --------------------------------
        # SAME preprocessing as training
        # --------------------------------

        skeleton = self.normalize_skeleton(
            skeleton
        )

        skeleton = self.temporal_resample(
            skeleton
        )

        X = torch.tensor(
            skeleton,
            dtype=torch.float32
        )

        return X, row["trial_id"]

In [24]:
from pathlib import Path

BASE = Path("/kaggle/input/datasets/samasiayushman/small-model-track/Testing/Testing/small_model_track_test-007/small_model_track_test")

for p in BASE.rglob("SM_test_0001"):
    print("Found:", p)

Found: /kaggle/input/datasets/samasiayushman/small-model-track/Testing/Testing/small_model_track_test-007/small_model_track_test/SM_test_0001


In [25]:
TEST_ROOT = p.parent
print(TEST_ROOT)

/kaggle/input/datasets/samasiayushman/small-model-track/Testing/Testing/small_model_track_test-007/small_model_track_test


In [26]:
TEST_ROOT = Path(
    "/kaggle/input/datasets/samasiayushman/"
    "small-model-track/Testing/Testing/"
    "small_model_track_test-007/small_model_track_test"
)

test_records = []

for trial_dir in sorted(TEST_ROOT.iterdir()):

    if not trial_dir.is_dir():
        continue

    test_records.append({
        "trial_id": trial_dir.name,
        "path": str(trial_dir),
        "skeleton": str(trial_dir),
    })

test_df = pd.DataFrame(test_records)

print("Test trials:", len(test_df))
print(test_df.head())

Test trials: 406
       trial_id                                               path  \
0       .claude  /kaggle/input/datasets/samasiayushman/small-mo...   
1  SM_test_0001  /kaggle/input/datasets/samasiayushman/small-mo...   
2  SM_test_0002  /kaggle/input/datasets/samasiayushman/small-mo...   
3  SM_test_0003  /kaggle/input/datasets/samasiayushman/small-mo...   
4  SM_test_0004  /kaggle/input/datasets/samasiayushman/small-mo...   

                                            skeleton  
0  /kaggle/input/datasets/samasiayushman/small-mo...  
1  /kaggle/input/datasets/samasiayushman/small-mo...  
2  /kaggle/input/datasets/samasiayushman/small-mo...  
3  /kaggle/input/datasets/samasiayushman/small-mo...  
4  /kaggle/input/datasets/samasiayushman/small-mo...  


In [27]:
test_dataset = SkeletonTestDataset(
    test_df,
    sequence_length=64
)

print("Test trials:", len(test_dataset))

Test trials: 406


In [28]:
X, trial_id = test_dataset[0]

print("Trial:", trial_id)
print("Shape:", X.shape)
print("dtype:", X.dtype)
print("Min:", X.min().item())
print("Max:", X.max().item())

Trial: .claude
Shape: torch.Size([64, 17, 3])
dtype: torch.float32
Min: 0.0
Max: 0.0


In [29]:
test_loader = DataLoader(
    test_dataset,
    batch_size=32,
    shuffle=False,
    num_workers=0,
    pin_memory=False
)

X, trial_ids = next(iter(test_loader))

print("Batch X:", X.shape)
print("Trial IDs:", trial_ids[:5])

Batch X: torch.Size([32, 64, 17, 3])
Trial IDs: ('.claude', 'SM_test_0001', 'SM_test_0002', 'SM_test_0003', 'SM_test_0004')


In [30]:
model.eval()

all_predictions = []
all_trial_ids = []

with torch.no_grad():

    for X, trial_ids in test_loader:

        X = X.to(device)

        logits = model(X)

        predictions = torch.argmax(logits, dim=1)

        all_predictions.extend(
            predictions.cpu().numpy()
        )

        all_trial_ids.extend(trial_ids)

In [31]:
print("Number of predictions:", len(all_predictions))
print("Number of trial IDs:", len(all_trial_ids))

print("\nFirst predictions:")
for trial_id, pred in zip(
    all_trial_ids[:10],
    all_predictions[:10]
):
    print(trial_id, "->", pred)

Number of predictions: 406
Number of trial IDs: 406

First predictions:
.claude -> 35
SM_test_0001 -> 35
SM_test_0002 -> 35
SM_test_0003 -> 35
SM_test_0004 -> 35
SM_test_0005 -> 35
SM_test_0006 -> 35
SM_test_0007 -> 35
SM_test_0008 -> 35
SM_test_0009 -> 35


In [32]:
from collections import Counter

prediction_counts = Counter(all_predictions)

print("Predicted classes:")
for label, count in sorted(prediction_counts.items()):
    print(f"{label:2d}: {count}")

Predicted classes:
35: 406


In [33]:
submission = pd.DataFrame({
    "path": [
        f"small_model_track_test/{trial_id}/"
        for trial_id in all_trial_ids
    ],
    "prediction": all_predictions
})

print(submission.head())
print(submission.shape)

print(submission.head())
print(submission.shape)
submission.to_csv("submission.csv",index=False)
print(submission.head())
print(submission.shape)

                                   path  prediction
0       small_model_track_test/.claude/          35
1  small_model_track_test/SM_test_0001/          35
2  small_model_track_test/SM_test_0002/          35
3  small_model_track_test/SM_test_0003/          35
4  small_model_track_test/SM_test_0004/          35
(406, 2)
                                   path  prediction
0       small_model_track_test/.claude/          35
1  small_model_track_test/SM_test_0001/          35
2  small_model_track_test/SM_test_0002/          35
3  small_model_track_test/SM_test_0003/          35
4  small_model_track_test/SM_test_0004/          35
(406, 2)
                                   path  prediction
0       small_model_track_test/.claude/          35
1  small_model_track_test/SM_test_0001/          35
2  small_model_track_test/SM_test_0002/          35
3  small_model_track_test/SM_test_0003/          35
4  small_model_track_test/SM_test_0004/          35
(406, 2)
